# 09_check_scoring

1 件の参考文献文字列に対して Solr 検索を行い、候補文献の `RC / CC / MC` と中間値を確認する。

## 確認内容

この notebook では以下を確認する。

1. 参考文献文字列から `R` を生成する
2. Solr / BM25 で候補文献を取得する
3. 候補文献ごとに `Cf` / `Cvf` を再生成する
4. まず `RC / CC / MC` を確認する
5. 必要な場合だけ CC の field 別スコアや `Cf / Cvf` の詳細を確認する

In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# 設定値・検索関数・スコア計算関数を読み込む
from pprint import pprint

from tanigawa_shoshi.config import (
    BASE_POSITIVE_EXAMPLES_PATH,
    NEGATIVE_EXAMPLES_PATH,
    POSITIVE_EXAMPLES_PATH,
)
from tanigawa_shoshi.evaluation_data import load_examples
from tanigawa_shoshi.scoring import (
    build_candidate_cc_token_sets,
    build_candidate_field_token_sets,
    build_reference_token_set,
    compute_cc,
    compute_mc,
    compute_rc,
)
from tanigawa_shoshi.search import search_reference


In [ ]:
# 確認対象データと検索条件を設定する
examples_path = BASE_POSITIVE_EXAMPLES_PATH
example_index = 0
rows = 100
preview_count = 10


In [ ]:
# 例データを読み込み、確認対象の 1 件を取り出す
examples = load_examples(examples_path)
example = examples[example_index]
reference_tokens = build_reference_token_set(example["reference_text"])

print(f"examples path: {examples_path}")
print(f"loaded examples: {len(examples)}")
print(f"example index: {example_index}")
print()
print("selected example")
pprint(
    {
        "example_id": example["example_id"],
        "label": example["label"],
        "doi": example["doi"],
        "style": example["style"],
        "field_names": example["field_names"],
        "reference_text": example["reference_text"],
    }
)
print()
print(f"reference token count: {len(reference_tokens)}")
print(reference_tokens)


In [ ]:
# Solr / BM25 で候補文献を取得する
search_result = search_reference(example["reference_text"], rows=rows)
results = list(search_result["results"])

print("search params")
pprint(search_result["params"])
print()
print(f"search tokens: {len(search_result['tokens'])}")
print(f"candidate count: {len(results)}")
print()

for index, candidate in enumerate(results[:preview_count], start=1):
    title_values = candidate.get("title") or []
    print("=" * 80)
    print(f"rank: {index}")
    print(f"doi: {candidate.get('doi')}")
    print(f"score: {candidate.get('score')}")
    print(f"title: {title_values[0] if title_values else ''}")


In [ ]:
# notebook 上で field 別の CC スコアも確認できるようにする補助関数
def normalize_cc_field_names(field_names):
    normalized = []
    for field_name in field_names:
        if field_name == "authors":
            normalized.append("first_author")
        else:
            normalized.append(field_name)
    return normalized


def compute_cc_field_scores(reference_tokens, candidate_field_tokens, candidate_field_variants, field_names):
    reference_token_set = set(reference_tokens)
    field_scores = {}
    for field_name in normalize_cc_field_names(field_names):
        variant_token_sets = candidate_field_variants.get(field_name) or []
        if variant_token_sets:
            score = max(
                len(reference_token_set & set(tokens)) / len(set(tokens))
                for tokens in variant_token_sets
                if set(tokens)
            )
        else:
            candidate_tokens = candidate_field_tokens.get(field_name) or []
            candidate_token_set = set(candidate_tokens)
            score = (
                len(reference_token_set & candidate_token_set) / len(candidate_token_set)
                if candidate_token_set
                else 0.0
            )
        field_scores[field_name] = score
    return field_scores


def score_candidate(example, candidate):
    candidate_field_tokens = build_candidate_field_token_sets(
        candidate,
        field_names=example["field_names"],
    )
    candidate_field_variants = build_candidate_cc_token_sets(
        candidate,
        field_names=example["field_names"],
    )
    rc = compute_rc(
        reference_tokens,
        candidate_field_tokens,
        field_names=example["field_names"],
    )
    cc = compute_cc(
        reference_tokens,
        candidate_field_tokens,
        candidate_field_variants,
        field_names=example["field_names"],
    )
    mc = compute_mc(rc, cc)
    cc_field_scores = compute_cc_field_scores(
        reference_tokens,
        candidate_field_tokens,
        candidate_field_variants,
        example["field_names"],
    )
    return {
        "doi": candidate.get("doi"),
        "title": (candidate.get("title") or [""])[0],
        "bm25_score": candidate.get("score"),
        "rc": rc,
        "cc": cc,
        "mc": mc,
        "candidate_field_tokens": candidate_field_tokens,
        "candidate_field_variants": candidate_field_variants,
        "cc_field_scores": cc_field_scores,
        "raw_candidate": candidate,
    }


In [ ]:
# 候補文献へ RC / CC / MC を付与し、上位を確認する
scored_candidates = [score_candidate(example, candidate) for candidate in results]

print(f"scored candidate count: {len(scored_candidates)}")
print()

for index, scored in enumerate(scored_candidates[:preview_count], start=1):
    print("=" * 100)
    print(f"rank: {index}")
    print(f"doi: {scored['doi']}")
    print(f"title: {scored['title']}")
    print(f"bm25_score: {scored['bm25_score']}")
    print(f"rc: {scored['rc']}")
    print(f"cc: {scored['cc']}")
    print(f"mc: {scored['mc']}")

if example["doi"]:
    gold_matches = [scored for scored in scored_candidates if scored["doi"] == example["doi"]]
    print()
    print(f"gold match count: {len(gold_matches)}")
    if gold_matches:
        print("gold match score")
        pprint(
            {
                "doi": gold_matches[0]["doi"],
                "title": gold_matches[0]["title"],
                "bm25_score": gold_matches[0]["bm25_score"],
                "rc": gold_matches[0]["rc"],
                "cc": gold_matches[0]["cc"],
                "mc": gold_matches[0]["mc"],
            }
        )


In [ ]:
# 1 候補を選んで、基本スコアだけを確認する
if example["doi"]:
    matching_candidates = [scored for scored in scored_candidates if scored["doi"] == example["doi"]]
    inspected_candidate = matching_candidates[0] if matching_candidates else scored_candidates[0]
else:
    inspected_candidate = scored_candidates[0]

print("inspected candidate")
pprint(
    {
        "doi": inspected_candidate["doi"],
        "title": inspected_candidate["title"],
        "bm25_score": inspected_candidate["bm25_score"],
        "rc": inspected_candidate["rc"],
        "cc": inspected_candidate["cc"],
        "mc": inspected_candidate["mc"],
    }
)


## 詳細確認（必要なときだけ実行）

通常確認では `RC / CC / MC` だけを見ればよい。
CC の field 別スコアや `Cf / Cvf` の中身を確認したい場合だけ、次のセルを実行する。

In [ ]:
# CC の field 別スコアと Cf / Cvf の詳細を確認する
print("cc_field_scores")
pprint(inspected_candidate["cc_field_scores"])
print()
print("candidate_field_tokens")
pprint(inspected_candidate["candidate_field_tokens"])
print()
print("candidate_field_variants")
pprint(inspected_candidate["candidate_field_variants"])
